# Market Basket Analysis Algorithms
# Apriori and FPGrowth

### Imports and Setup

In [1]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth
import time

games_df = pd.read_csv("gameDim.csv", encoding="latin1")
fact_table = pd.read_csv("steamUserFact.csv")

fact_table = fact_table.drop_duplicates()
games_df.head()

games_df = games_df.drop(columns=["specs", "release_date", "price", "genres", "tags"])
fact_table = fact_table[fact_table["playtime_hours"] > 2]

### Removing Rarely Played Games

In [2]:
final = pd.merge(fact_table, games_df, on="game_id", how="left")
transactions = []

for i, j in final.groupby("steam_id"):
    transactions.append(list(j["game_name"]))

### Transactions

In [3]:
transactions[:2]

transaction_enc = TransactionEncoder()
encoded = transaction_enc.fit_transform(transactions)

### Transaction Encoder

In [4]:
bask = pd.DataFrame(encoded, columns=transaction_enc.columns_)
bask.head()

,"""Glow Ball"" - The billiard puzzle game",//N.P.P.D. RUSH//- The milk of Ultraviolet,//SNOWFLAKE TATTOO//,001 Game Creator,0RBITALIS,10 Second Ninja,10 Second Ninja X,10 Years After,"10,000,000",100% Orange Juice,...,rFactor,rFactor 2,realMYST,realMyst: Masterpiece Edition,resident evil 4 / biohazard 4,rymdkapsel,sZone-Online,the static speaks my name,theHunter Classic,theHunter: Primal
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


### Apriori

In [5]:
start = time.time()
freq_item_set = apriori(bask, min_support=0.02, use_colnames=True)
end = time.time()

end - start

freq_item_set.sample(5)

rules_apriori = association_rules(
    freq_item_set, 
    metric='lift', 
    min_threshold=1
)[["antecedents", "consequents", "support", "confidence", "lift"]]

rules_apriori.sort_values(by="lift", ascending=False).head(20)

,antecedents,consequents,support,confidence,lift
489,frozenset({Half-Life 2: Episode Two}),frozenset({Half-Life 2: Episode One}),0.020128,0.645614,21.427488
488,frozenset({Half-Life 2: Episode One}),frozenset({Half-Life 2: Episode Two}),0.020128,0.668050,21.427488
475,frozenset({Half-Life 2}),frozenset({Half-Life 2: Episode One}),0.025332,0.219826,7.295888
474,frozenset({Half-Life 2: Episode One}),frozenset({Half-Life 2}),0.025332,0.840768,7.295888
477,frozenset({Half-Life 2: Episode Two}),frozenset({Half-Life 2}),0.025817,0.828070,7.185704
476,frozenset({Half-Life 2}),frozenset({Half-Life 2: Episode Two}),0.025817,0.224030,7.185704
1704,frozenset({The Binding of Isaac: Rebirth}),"frozenset({The Binding of Isaac, Terraria})",0.020707,0.335953,6.514339
1701,"frozenset({The Binding of Isaac, Terraria})",frozenset({The Binding of Isaac: Rebirth}),0.020707,0.401515,6.514339
845,frozenset({Arma 3}),"frozenset({DayZ, Arma 2: Operation Arrowhead})",0.027270,0.291952,6.419845
840,"frozenset({DayZ, Arma 2: Operation Arrowhead})",frozenset({Arma 3}),0.027270,0.599656,6.419845


Episode One and Two of a game are recommended together, showing that the algorithm is working.

### FPGrowth

In [6]:
fp_itemset = fpgrowth(bask, min_support=0.02, use_colnames=True)

rules_fp = association_rules(
    fp_itemset, 
    metric='lift', 
    min_threshold=1
)[["antecedents", "consequents", "support", "confidence", "lift"]]

rules_fp.sort_values(by="lift", ascending=False).head(20)

,antecedents,consequents,support,confidence,lift
725,frozenset({Half-Life 2: Episode Two}),frozenset({Half-Life 2: Episode One}),0.020128,0.645614,21.427488
724,frozenset({Half-Life 2: Episode One}),frozenset({Half-Life 2: Episode Two}),0.020128,0.668050,21.427488
723,frozenset({Half-Life 2}),frozenset({Half-Life 2: Episode One}),0.025332,0.219826,7.295888
722,frozenset({Half-Life 2: Episode One}),frozenset({Half-Life 2}),0.025332,0.840768,7.295888
721,frozenset({Half-Life 2: Episode Two}),frozenset({Half-Life 2}),0.025817,0.828070,7.185704
720,frozenset({Half-Life 2}),frozenset({Half-Life 2: Episode Two}),0.025817,0.224030,7.185704
1074,frozenset({The Binding of Isaac: Rebirth}),"frozenset({The Binding of Isaac, Terraria})",0.020707,0.335953,6.514339
1071,"frozenset({The Binding of Isaac, Terraria})",frozenset({The Binding of Isaac: Rebirth}),0.020707,0.401515,6.514339
1112,"frozenset({DayZ, Arma 2: Operation Arrowhead})",frozenset({Arma 3}),0.027270,0.599656,6.419845
1117,frozenset({Arma 3}),"frozenset({DayZ, Arma 2: Operation Arrowhead})",0.027270,0.291952,6.419845


This shows the same results as Apriori, showing that this algorithm is also working.

### Comparison

In [7]:
def ev(freq_i, name):
    start = time.time()
    rules = association_rules(
        freq_i, 
        metric='lift', 
        min_threshold=1
    )[["antecedents", "consequents", "support", "confidence", "lift"]]
    end = time.time()
    
    return {
        "model": name,
        "Rules Length": len(rules),
        "Time Taken": end - start,
        "Average Support": rules["support"].mean(),
        "Average Confidence": rules["confidence"].mean(),
        "Average Lift": rules["lift"].mean()
    }

apriori = ev(freq_item_set, "Apriori")
fp = ev(fp_itemset, "FP_Growth")

pd.DataFrame([apriori, fp])

,model,Rules Length,Time Taken,Average Support,Average Confidence,Average Lift
0,Apriori,1730,0.011580,0.030842,0.265218,1.745984
1,FP_Growth,1730,0.010721,0.030842,0.265218,1.745984


FPGrowth is faster as shown above.